# Análise exploratória tabela estabelecimentos RAIS

## Setups

### Definição de constantes

In [1]:
GCP_PROJECT_ID = "pure-league-482018-a7"
DATASET_NAME = "basedosdados"
SCHEMA_NAME = "br_me_rais"
TABLE= "microdados_vinculos"

### Configurações de ambiente

In [2]:
from google.cloud import bigquery
from pathlib import Path
import os
import warnings

actual_path = Path().absolute()
os.chdir(actual_path.parent.parent.parent)

warnings.filterwarnings("ignore", category=UserWarning, module="google.cloud.bigquery")
client = bigquery.Client(project=GCP_PROJECT_ID)

## análise de metadados

In [3]:
query_metadata_t1 = f"""
    SELECT *
    FROM `{DATASET_NAME}.{SCHEMA_NAME}.INFORMATION_SCHEMA.COLUMNS`
    WHERE table_name = '{TABLE}';
"""

df_metadata_t1 = client.query(query_metadata_t1).to_dataframe()
df_metadata_t1

,table_catalog,table_schema,table_name,column_name,ordinal_position,is_nullable,data_type,is_generated,generation_expression,is_stored,...,is_system_defined,is_partitioning_column,clustering_ordinal_position,collation_name,column_default,rounding_mode,data_policies,data_governance_tags,policy_tags,async_generation_status
0,basedosdados,br_me_rais,microdados_vinculos,ano,1,YES,INT64,NEVER,NaN,NaN,...,NO,YES,<NA>,NULL,NULL,NaN,[],[],[],None
1,basedosdados,br_me_rais,microdados_vinculos,sigla_uf,2,YES,STRING,NEVER,NaN,NaN,...,NO,NO,1,NULL,NULL,NaN,[],[],[],None
2,basedosdados,br_me_rais,microdados_vinculos,id_municipio,3,YES,STRING,NEVER,NaN,NaN,...,NO,NO,2,NULL,NULL,NaN,[],[],[],None
3,basedosdados,br_me_rais,microdados_vinculos,tipo_vinculo,4,YES,STRING,NEVER,NaN,NaN,...,NO,NO,<NA>,NULL,NULL,NaN,[],[],[],None
4,basedosdados,br_me_rais,microdados_vinculos,vinculo_ativo_3112,5,YES,STRING,NEVER,NaN,NaN,...,NO,NO,<NA>,NULL,NULL,NaN,[],[],[],None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62,basedosdados,br_me_rais,microdados_vinculos,bairros_sp,63,YES,STRING,NEVER,NaN,NaN,...,NO,NO,<NA>,NULL,NULL,NaN,[],[],[],None
63,basedosdados,br_me_rais,microdados_vinculos,distritos_sp,64,YES,STRING,NEVER,NaN,NaN,...,NO,NO,<NA>,NULL,NULL,NaN,[],[],[],None
64,basedosdados,br_me_rais,microdados_vinculos,bairros_fortaleza,65,YES,STRING,NEVER,NaN,NaN,...,NO,NO,<NA>,NULL,NULL,NaN,[],[],[],None
65,basedosdados,br_me_rais,microdados_vinculos,bairros_rj,66,YES,STRING,NEVER,NaN,NaN,...,NO,NO,<NA>,NULL,NULL,NaN,[],[],[],None


## Análise de preenchimento

In [4]:
count_list = [f"COUNTIF({col} IS NOT NULL) / COUNT(*) AS {col}_filled_pct" for col in df_metadata_t1['column_name'].values]
count_list_str = ",\n    ".join(count_list)

query_missingness = f"""
SELECT
    {count_list_str}
FROM 
`basedosdados.br_me_rais.{TABLE}`
"""

df2 = client.query(query_missingness).to_dataframe()
df2_long = df2.T.reset_index()
df2_long.columns = ["field", "value"]
df2_long.sort_values(by="value", ascending=False)

,field,value
0,ano_filled_pct,1.000000
1,sigla_uf_filled_pct,1.000000
4,vinculo_ativo_3112_filled_pct,1.000000
53,sexo_filled_pct,1.000000
14,tempo_emprego_filled_pct,1.000000
...,...,...
41,subatividade_ibge_filled_pct,0.135153
64,bairros_fortaleza_filled_pct,0.128802
21,indicador_vinculo_abandonado_filled_pct,0.121290
40,valor_salario_contratual_filled_pct,0.030604


#### Registros a partir de 2020

In [5]:
count_list = [f"COUNTIF({col} IS NOT NULL) / COUNT(*) AS {col}_filled_pct" for col in df_metadata_t1['column_name'].values]
count_list_str = ",\n    ".join(count_list)

query_missingness = f"""
SELECT
    {count_list_str}
FROM 
`basedosdados.br_me_rais.{TABLE}`
WHERE ano >= 2020
"""

df2 = client.query(query_missingness).to_dataframe()
df2_long = df2.T.reset_index()
df2_long.columns = ["field", "value"]
df2_long.sort_values(by="value", ascending=False)

,field,value
0,ano_filled_pct,1.0
1,sigla_uf_filled_pct,1.0
3,tipo_vinculo_filled_pct,1.0
59,tipo_estabelecimento_filled_pct,1.0
4,vinculo_ativo_3112_filled_pct,1.0
...,...,...
50,grau_instrucao_1985_2005_filled_pct,0.0
39,tipo_salario_filled_pct,0.0
43,cbo_1994_filled_pct,0.0
40,valor_salario_contratual_filled_pct,0.0


## análise de distribuição de valores

### Valores Categoricos

In [6]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import webbrowser


def get_value_counts(client, table, col):
    query = f"""
    SELECT
        `{col}` AS value,
        COUNT(*) AS freq
    FROM `{table}`
    GROUP BY `{col}`
    ORDER BY freq DESC
    
    """
    df = client.query(query).to_dataframe()
    df["value"] = df["value"].astype(str)  # garante eixo categórico consistente
    return df

dtypes=list(df_metadata_t1['data_type'].values)
columns = list(df_metadata_t1['column_name'].values)

cat_columns=[
    col for col, dtype in zip(columns, dtypes) 
    if dtype in ['STRING',"INT64", 'DATE', 'DATETIME', 'TIMESTAMP']
    ]

table = f"{DATASET_NAME}.{SCHEMA_NAME}.{TABLE}"

# Coleta os value_counts de cada coluna (uma query por coluna)
value_counts_dict = {}
for col in cat_columns:
    print(f"Consultando: {col}")
    value_counts_dict[col] = get_value_counts(client, table, col)

# Monta o grid vertical: 1 gráfico por linha
n = len(cat_columns)
fig = make_subplots(
    rows=n, cols=1,
    subplot_titles=[f"Distribuição: {col}" for col in cat_columns],
    vertical_spacing=0.4 / n  # espaçamento proporcional para não sobrepor títulos
)

for i, col in enumerate(cat_columns, start=1):
    df_vc = value_counts_dict[col]
    fig.add_trace(
        go.Bar(
            x=df_vc["value"],
            y=df_vc["freq"],
            name=col,
            showlegend=False
        ),
        row=i, col=1
    )
    fig.update_xaxes(tickangle=45, row=i, col=1)

# Altura por gráfico ~350px, permitindo scroll vertical no notebook/HTML
fig.update_layout(
    height=350 * n,
    width=1000,
    title_text=f"Distribuição de frequência por variável da tabela '{TABLE}'",
    showlegend=False
)

# Salva como HTML
output_path = os.path.abspath(f"data/outputs/distribuicao_{TABLE}_cat_columns.html")
fig.show()

Consultando: ano
Consultando: sigla_uf
Consultando: id_municipio
Consultando: tipo_vinculo
Consultando: vinculo_ativo_3112
Consultando: tipo_admissao
Consultando: mes_admissao
Consultando: mes_desligamento
Consultando: motivo_desligamento
Consultando: causa_desligamento_1
Consultando: causa_desligamento_2
Consultando: causa_desligamento_3
Consultando: faixa_tempo_emprego
Consultando: faixa_horas_contratadas
Consultando: quantidade_horas_contratadas
Consultando: id_municipio_trabalho
Consultando: quantidade_dias_afastamento
Consultando: indicador_cei_vinculado
Consultando: indicador_trabalho_parcial
Consultando: indicador_trabalho_intermitente
Consultando: indicador_vinculo_abandonado
Consultando: faixa_remuneracao_media_sm
Consultando: faixa_remuneracao_dezembro_sm
Consultando: tipo_salario
Consultando: subatividade_ibge
Consultando: subsetor_ibge
Consultando: cbo_1994
Consultando: cbo_2002
Consultando: cnae_1
Consultando: cnae_2
Consultando: cnae_2_subclasse
Consultando: faixa_etaria


### Análise colunas numericas

In [7]:
from scripts.num_dist_vizs import *

### Histograma

In [8]:


MAX_TICKS = 50
N_BINS = 100  # numero de bins dentro da janela exibida (média ± N_STDDEV_RANGE*σ)
 
dtypes = list(df_metadata_t1['data_type'].values)
columns = list(df_metadata_t1['column_name'].values)
 
num_columns = [
    col for col, dtype in zip(columns, dtypes)
    if dtype in ['FLOAT64']
]
 
table = f"{DATASET_NAME}.{SCHEMA_NAME}.{TABLE}"
 
# Coleta estatísticas e histogramas de cada coluna numérica
histogram_dict, stats_dict, valid_columns, bin_width_dict = collect_stats_and_histograms(
    client, table, num_columns, n_bins=N_BINS
)
 
# Monta o gráfico
fig, valid_columns = build_histogram_figure(
    histogram_dict, stats_dict, bin_width_dict, TABLE,
    n_bins=N_BINS, max_ticks=MAX_TICKS
)

fig.show()

Consultando estatísticas: tempo_emprego
Consultando histograma: tempo_emprego (bin_width=3.448, janela=[0.1, 344.9])
Consultando estatísticas: valor_remuneracao_media_sm
Consultando histograma: valor_remuneracao_media_sm (bin_width=0.25, janela=[0, 25])
Consultando estatísticas: valor_remuneracao_media
Consultando histograma: valor_remuneracao_media (bin_width=155.6, janela=[0, 1.556e+04])
Consultando estatísticas: valor_remuneracao_dezembro_sm
Consultando histograma: valor_remuneracao_dezembro_sm (bin_width=0.2448, janela=[0, 24.48])
Consultando estatísticas: valor_remuneracao_janeiro
Consultando histograma: valor_remuneracao_janeiro (bin_width=177, janela=[0, 1.77e+04])
Consultando estatísticas: valor_remuneracao_fevereiro
Consultando histograma: valor_remuneracao_fevereiro (bin_width=171.9, janela=[0, 1.719e+04])
Consultando estatísticas: valor_remuneracao_marco
Consultando histograma: valor_remuneracao_marco (bin_width=172.6, janela=[0, 1.726e+04])
Consultando estatísticas: valor_r

### Distribuição contínua

In [9]:

N_STDDEV_RANGE = 4
MAX_TICKS = 50

 
dtypes = list(df_metadata_t1['data_type'].values)
columns = list(df_metadata_t1['column_name'].values)
 
num_columns = [
    col for col, dtype in zip(columns, dtypes)
    if dtype in ['FLOAT64']
]
 
table = f"{DATASET_NAME}.{SCHEMA_NAME}.{TABLE}"
 
# Coleta estatísticas e histogramas de cada coluna numérica
bins_dict, stats_dict, window_dict, valid_columns = collect_stats_and_bins(
    client, table, num_columns, n_bins=200
)

fig, valid_columns = build_density_figure(
    bins_dict, stats_dict, window_dict, TABLE, log_vals=True
)
 
# Salva como HTML e abre no navegador
output_path = os.path.abspath(f"data/outputs/html/distribuicao_{TABLE}_num_columns.html")
fig.show()
 

### Box Plot

In [10]:
stats_dict, valid_columns = collect_boxplot_stats(client, table, num_columns)

fig, valid_columns = build_boxplot_figure(stats_dict, TABLE, log_vals=False)
output_path = os.path.abspath(f"data/outputs/html/distribuicao_boxplot_{TABLE}_num_columns.html")
fig.show()
 

Consultando quartis: tempo_emprego
Consultando quartis: valor_remuneracao_media_sm
Consultando quartis: valor_remuneracao_media
Consultando quartis: valor_remuneracao_dezembro_sm
Consultando quartis: valor_remuneracao_janeiro
Consultando quartis: valor_remuneracao_fevereiro
Consultando quartis: valor_remuneracao_marco
Consultando quartis: valor_remuneracao_abril
Consultando quartis: valor_remuneracao_maio
Consultando quartis: valor_remuneracao_junho
Consultando quartis: valor_remuneracao_julho
Consultando quartis: valor_remuneracao_agosto
Consultando quartis: valor_remuneracao_setembro
Consultando quartis: valor_remuneracao_outubro
Consultando quartis: valor_remuneracao_novembro
Consultando quartis: valor_remuneracao_dezembro
Consultando quartis: valor_salario_contratual
